# K-mer Feature Pipeline (Binary presence)

This notebook performs k-mer feature selection and trains models using binary presence/absence
features for selection and final modeling. The prevalence-first approach keeps memory use low.
Models and vocabulary are saved to
`output/.`

In [1]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import joblib

## Helper functions
These helpers iterate per-genome k-mer dump files, build prevalence counts, construct
sparse matrices (binary mode), and load phenotype labels.

In [29]:
# Helper utilities for k-mer pipelines (binary-presence notebook)
# Each function below includes a short comment/docstring explaining its role, inputs, and outputs.
def iter_genome_kmers(dump_path: Path) -> dict[str, int]:
    """Read a per-genome k-mer dump and return a dict of {kmer: count}.

    Parameters:
    - dump_path: Path to a single genome's k-mer dump file (two columns: kmer count).

    Returns:
    - dict mapping kmer (str) to integer count.
    """
    kmers = {}
    with dump_path.open('r', encoding='utf8', errors='ignore') as fh:
        for line in fh:
            parts = line.strip().split()
            if len(parts) != 2:
                continue
            kmer, cnt = parts
            try:
                kmers[kmer] = int(cnt)
            except ValueError:
                continue
    return kmers

def build_prevalence(dump_dir: str, genome_ids: list[str]) -> Counter:
    """Aggregate presence counts for each k-mer across a list of genomes.

    Scans each genome's k-mer dump and counts how many genomes contain each k-mer (presence, not counts).

    Parameters:
    - dump_dir: directory containing `{GenomeID}_db_kmers.txt` files.
    - genome_ids: list of genome identifiers to include.

    Returns:
    - Counter where keys are k-mers and values are the number of genomes containing that k-mer.
    """
    prevalence = Counter()
    dump_dir = Path(dump_dir)
    for gid in genome_ids:
        dump_path = dump_dir/f'{gid}_db_kmers.txt'
        if not dump_path.exists():
            continue
        kmers = iter_genome_kmers(dump_path)
        # update counts by presence (keys only)
        prevalence.update(kmers.keys())
    return prevalence

def select_vocab_by_prevalence(prevalence: Counter, n_genomes: int,
                                   min_frac: float = 0.02, max_frac: float = 0.95) -> list[str]:
    """Filter k-mers by prevalence across genomes to remove extremely rare or ubiquitous features.

    Parameters:
    - prevalence: Counter from `build_prevalence` mapping k-mer -> genome occurrence count.
    - n_genomes: number of genomes used to compute prevalence (for fraction -> absolute conversion).
    - min_frac: minimum fraction of genomes that must contain a k-mer to keep it.
    - max_frac: maximum fraction of genomes that may contain a k-mer to keep it.

    Returns:
    - List of k-mer strings that pass the prevalence filters.
    """
    min_count = int(np.ceil(min_frac * n_genomes))
    max_count = int(np.floor(max_frac * n_genomes))
    vocab = [k for k, c in prevalence.items() if min_count <= c <= max_count]
    return vocab

def build_sparse_matrix(
    dump_dir: str,
    genome_ids: list[str],
    vocab: list[str],
    binary: bool = True,
    chunk_size: int = 200
    ) -> sparse.csr_matrix:
    """Construct a sparse (CSR) matrix of shape (n_genomes, n_features) from k-mer dumps.

    This reads each genome's dump and populates the matrix using the provided `vocab` index.
    When `binary` is True, presence is recorded as 1; otherwise counts are used (int).
    The build is chunked to keep peak memory low.

    Parameters:
    - dump_dir: directory with per-genome k-mer dump files.
    - genome_ids: ordered list of genome IDs corresponding to rows in the matrix.
    - vocab: ordered list of k-mers corresponding to columns in the matrix.
    - binary: whether to collapse counts to binary presence/absence.
    - chunk_size: number of genomes per chunk when building the matrix.

    Returns:
    - scipy.sparse.csr_matrix with dtype `np.int8` for binary (or `np.int32` for counts).
    """
    vocab_index = {kmer: idx for idx, kmer in enumerate(vocab)}
    dump_dir = Path(dump_dir)
    blocks = []
    dtype = np.int8 if binary else np.int32
    n_features = len(vocab)

    for start in range(0, len(genome_ids), chunk_size):
        chunk_ids = genome_ids[start:start + chunk_size]
        rows = []
        cols = []
        data = []
        for row_idx, gid in enumerate(chunk_ids):
            dump_path = dump_dir / f'{gid}_db_kmers.txt'
            if not dump_path.exists():
                continue
            kmers = iter_genome_kmers(dump_path)
            for kmer in kmers.keys():
                col_idx = vocab_index.get(kmer)
                if col_idx is None:
                    continue
                rows.append(row_idx)
                cols.append(col_idx)
                data.append(1 if binary else int(kmers.get(kmer, 0)))
        if rows:
            block = sparse.csr_matrix((data, (rows, cols)), shape=(len(chunk_ids), n_features), dtype=dtype)
        else:
            block = sparse.csr_matrix((len(chunk_ids), n_features), dtype=dtype)
        blocks.append(block)
    if not blocks:
        return sparse.csr_matrix((0, n_features), dtype=dtype)
    return sparse.vstack(blocks, format='csr')

def load_labels(labels_path: Path) -> pd.Series:
    """Load phenotype labels from a CSV and return a binary series indexed by GenomeID.

    Expects a CSV with at least `GenomeID` and `phenotype` columns. Optionally maps 'I' -> 'R'.

    Parameters:
    - labels_path: Path to CSV containing labels.
    - treat_intermediate_as_resistant: if True, map 'I' to 'R' before filtering.

    Returns:
    - pandas Series indexed by GenomeID with values 1 for resistant and 0 for susceptible.
    """
    df = pd.read_csv(labels_path)
    if 'Genome ID' not in df.columns or 'phenotype' not in df.columns:
        raise ValueError('labels CSV must contain Genome ID and phenotype columns')
    pheno = df.set_index('Genome ID')['phenotype']
    return pheno

## Run Feature selection and Train models
Adjust the paths below (`dump_dir`, `labels_path`, `genome_ids_path`) if your files are elsewhere, then run this cell.

In [7]:
# define Paths
dump_dir = Path('../data/counted_kmers')
# expects columns: GenomeID, phenotype (R/S) and would have to adjust per antibiotics
labels_path = Path('../data/phenotype/ampicillin_phenotype.csv')  
genome_ids_path = labels_path


In [12]:
# Load genome ids and labels
genome_ids = []
if genome_ids_path.exists():
    id =pd.read_csv(genome_ids_path)
    genome_ids = id['Genome ID'].tolist()

y = load_labels(labels_path)
# genome_ids = [gid for gid in genome_ids if gid in y.index]
print(f'Using {len(genome_ids)} genomes for selection')


AttributeError: Can only use .str accessor with string values!

#### Applying Prevalence Filtering

In [ ]:
# Prevalence counting
prevalence = build_prevalence(dump_dir, genome_ids)
print(f'Unique k-mers observed: {len(prevalence):,d}')

In [ ]:
# Prevalence filter, mmin_frac=0.2 means we keep k-mers present in at least 20% of genomes, 
# max_frac=0.95 means we exclude k-mers present in more than 95% of genomes. This helps remove very rare k-mers (which may be noise) 
# and very common k-mers (which may not be informative for classification).

# the value of min_frac and max_frac is dependent on the number of genome present
vocab = select_vocab_by_prevalence(prevalence, n_genomes=len(genome_ids), min_frac=0.2, max_frac=0.95)
print(f'Vocab after prevalence filter: {len(vocab):,d}')

In [27]:
# Quick RAM estimate using a small genome sample
sample_size = min(100, len(genome_ids))
sample_ids = genome_ids[:sample_size]
vocab_set = set(vocab)
nnz_sample = 0
for gid in sample_ids:
    dump_path = dump_dir / f'{gid}_db_kmers.txt'
    if not dump_path.exists():
        continue
    kmers = iter_genome_kmers(dump_path)
    nnz_sample += sum(1 for k in kmers.keys() if k in vocab_set)

if sample_size == 0:
    raise ValueError('No genomes available for sampling.')

avg_k = nnz_sample / sample_size
nnz_est = int(avg_k * len(genome_ids))
csr_bytes = nnz_est * 5 + (len(genome_ids) + 1) * 4
csr_gb = csr_bytes / (1024 ** 3)
list_overhead_gb = csr_gb * 4
print(f'Avg kept k-mers per genome (sample): {avg_k:,.0f}')
print(f'Estimated nnz: {nnz_est:,d}')
print(f'Approx CSR size: {csr_gb:.2f} GB')
print(f'Rough list overhead (pre-CSR build): {list_overhead_gb:.2f} GB')
print('Rule of thumb: aim for total RAM >= CSR + list overhead + model buffers.')

Avg kept k-mers per genome (sample): 327,609
Estimated nnz: 1,839,198,610
Approx CSR size: 8.56 GB
Rough list overhead (pre-CSR build): 34.26 GB
Rule of thumb: aim for total RAM >= CSR + list overhead + model buffers.


#### Build Sparse Matrix

In [ ]:
# Build binary sparse matrix for selection
X_bin = build_sparse_matrix(dump_dir, genome_ids, vocab, binary=True)
y_arr = y.loc[genome_ids].to_numpy()
print('Built binary matrix for selection:', X_bin.shape)

##### Applying Chi Square selection

In [ ]:
# Chi-square selection
top_k = min(100000, X_bin.shape[1])
scores, _ = chi2(X_bin, y_arr)
top_idx = np.argsort(scores)[::-1][:top_k]
vocab_sel = [vocab[i] for i in top_idx]
print(f'Selected top k-mers: {len(vocab_sel):,d}')


### Build final binary sparse data and Train model

In [ ]:
# Build final binary matrix for modeling (only selected features)
X_final = build_sparse_matrix(dump_dir, genome_ids, vocab_sel, binary=True)
print('Built final binary matrix:', X_final.shape)


In [ ]:
# Train and evaluate
X_train, X_test, y_train, y_test = train_test_split(X_final, y_arr, test_size=0.2, random_state=42, stratify=y_arr)
log = LogisticRegression(max_iter=1000, solver='saga')
log.fit(X_train, y_train)
y_pred = log.predict(X_test)
print('Logistic accuracy (binary):', accuracy_score(y_test, y_pred))

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print('RF accuracy (binary):', accuracy_score(y_test, y_pred_rf))

# 7) Save artifacts
model_dir = Path('output/models')
out_dir = Path('output/feature_selection')
# out_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(vocab_sel, out_dir / 'vocab_selected_binary.pkl')
joblib.dump(log, model_dir / 'logistic_binary.joblib')
joblib.dump(rf, model_dir / 'rf_binary.joblib')
with open(out_dir / 'results_binary.json', 'w') as fh:
    json.dump({'logistic_acc': accuracy_score(y_test, y_pred), 'rf_acc': accuracy_score(y_test, y_pred_rf)}, fh, indent=2)
print('Saved selected vocabulary and models to:  output/')